# Análise Exploratória Inicial do Dataset LIAR
## Objetivo

Carregamento, transformação e análise exploratória do LIAR (6 classes).

## Declaração de uso de Inteligência Artificial

O código e descrição foram gerados e adaptados com auxílio de IA (ChatGPT, OpenAI).
## Etapas implementadas

**Carregamento dos dados**

- Utilização do dataset LIAR como no trabalho [1], e disponibilizado pela Universidade da Califórnia, Santa Bárbara (UCSB) [2].
- Adaptação do carregamento para utilizar os arquivos originais TSV, evitando incompatibilidades com versões recentes da biblioteca `datasets`.
- Preservação das partições originais de treinamento, validação e teste.

**Transformação dos rótulos**

- LIAR: manutenção das seis classes originais.
- Criação de um DataFrame contendo os textos e as classificações numéricas para comparação.

**Pré-processamento textual**

- Tratamento de valores ausentes.
- Conversão dos textos para minúsculas.
- Remoção de URLs, números, pontuação e caracteres especiais.
- Tokenização por espaços.
- Remoção de stopwords em inglês e palavras com menos de três caracteres.
- Reconstrução do texto processado para análise.
O texto original foi preservado em uma coluna separada.

**Visualização dos dados**

- Geração de nuvens de palavras para cada classe do dataset.
- Identificação das classes por número e nome.
- Aplicação de cores distintas para facilitar a interpretação visual.
- Organização das nuvens lado a lado e exportação das figuras em formato JPG.

## Considerações

Este notebook possui finalidade apenas de estudo e de experimentação. As nuvens de palavras representam frequências de termos e não constituem, isoladamente, evidência de capacidade discriminativa ou desempenho de classificação.

## Referências
[1] Wang, William Yang. "“Liar, liar pants on fire”: A new benchmark dataset for fake news detection." Proceedings of the 55th Annual Meeting of the Association for Computational Linguistics (Volume 2: Short Papers). 2017.

[2] [LIAR — Hugging Face](https://huggingface.co/datasets/ucsbai/liar)


# Implementação

In [ ]:
%pip install -q pandas matplotlib nltk wordcloud requests

## 1. INSTALAÇÃO E IMPORTAÇÕES

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import nltk
import pandas as pd
import requests

from nltk.corpus import stopwords
from wordcloud import WordCloud
nltk.download("stopwords", quiet=True)
import io
import zipfile
import requests
from pathlib import Path

SEED = 42
STOPWORDS = set(stopwords.words("english"))

DATASET_URL = "https://datasets-server.huggingface.co"
DATASET_NAME = "ucsbai/liar"
LIAR_URL = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"

## 2. DEFINIÇÃO DAS CLASSES

In [ ]:
LIAR_CLASSES = {"pants-fire": 0,"false": 1,"barely-true": 2,"half-true": 3, "mostly-true": 4,"true": 5,}

CLASS_NAMES = {
    "liar": {0: "Pants-fire", 1: "False", 2: "Barely-true", 3: "Half-true", 4: "Mostly-true", 5: "True",},
    
}

DATASET_MAPS = {
    "liar": LIAR_CLASSES,
    
}

# Definição dos tons de cinza por classe para cada dataset

CLASS_GRAY_COLORS = {
    "liar": {
        0: "#252525",  # Pants-fire
        1: "#525252",  # False
        2: "#737373",  # Barely-true
        3: "#969696",  # Half-true
        4: "#636363",  # Mostly-true
        5: "#404040",  # True
    }
}

## 3. CARREGAMENTO DO LIAR — ARQUIVOS ORIGINAIS

In [ ]:
def load_liar():
    response = requests.get(LIAR_URL, timeout=120)
    response.raise_for_status()
    files = {"train": "train.tsv", "validation": "valid.tsv", "test": "test.tsv",}
    splits = {}

    with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
        for split, filename in files.items():
            path = next(
                name for name in archive.namelist()
                if name.endswith(filename)
            )
            with archive.open(path) as file:
                df = pd.read_csv(
                    file,
                    sep="\t",
                    header=None,
                    dtype=str,
                )
            # Coluna 1: rótulo; coluna 2: afirmação.
            df = df.iloc[:, [2, 1]].copy()
            df.columns = ["statement", "original_label"]
            df["original_label"] = (
                df["original_label"]
                .str.strip()
                .str.lower()
            )
            splits[split] = df.reset_index(drop=True)
    return splits

splits = load_liar()
print("LIAR carregado com sucesso!")
for split, df in splits.items():
    print(f"{split}: {len(df)} registros")

## 4. TRANSFORMAÇÃO DOS DATASETS

In [ ]:
def create_label_columns(df):
    """Adiciona os rótulos numéricos ao mesmo texto."""
    result = df.copy()
    # Normaliza nomes, sem alterar os textos originais.
    names = (
        result["original_label"]
        .str.lower()
        .str.strip()
        .str.replace("_", "-", regex=False)
    )
    for dataset, mapping in DATASET_MAPS.items():
        result[dataset] = names.map(mapping)
        if result[dataset].isna().any():
            unknown = names[result[dataset].isna()].unique()
            raise ValueError(
                f"Rótulos não reconhecidos em {dataset}: {unknown}"
            )
        result[dataset] = result[dataset].astype(int)
    return result

splits = {
    name: create_label_columns(df)
    for name, df in splits.items()
}

# DataFrame completo para análise exploratória.
df_liar = pd.concat(
    splits.values(),
    ignore_index=True,
)

columns = ["statement", "original_label", "liar"]
classes = ["true", "pants-fire", "half-true"]
max_words = 8
indices = [2411, 7653, 392] #textos curtos de exemplos
print("\nExemplo de textos e classes transformadas:")
print(df_liar.loc[indices, columns].to_string(index=True))
#df_short = df_liar[df_liar["original_label"].isin(classes) & (df_liar["statement"].fillna("").str.split().str.len() <= max_words)]
#print(f"\nExemplo de textos com até {max_words} palavras por classe:")
#display(df_short.groupby("original_label", group_keys=False).sample(n=1, random_state=42)[columns])

## 5. DISTRIBUIÇÃO E BALANCEAMENTO DAS CLASSES

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def plot_class_distribution_all(df):
    """
    Plota a distribuição das classes dos datasets lado a lado.

    - Utiliza os mesmos tons de cinza das nuvens de palavras.
    - Cada classe possui sua própria tonalidade.
    - Exibe quantidade e percentual de amostras.
    - Exporta o gráfico em JPG com 300 dpi.
    """
    datasets = list(DATASET_MAPS.keys())

    fig, axes = plt.subplots(
        1,
        len(datasets),
        figsize=(12, 5.5),
        squeeze=False
    )
    axes = axes.flatten()

    for ax, dataset in zip(axes, datasets):
        class_names = CLASS_NAMES[dataset]
        # Contagem de amostras por classe
        counts = (
            df[dataset]
            .value_counts()
            .reindex(class_names.keys(), fill_value=0)
        )
        total = counts.sum()
        percentages = counts / total * 100
        # Identificação das classes
        labels = [
            f"{class_id}\n{class_name}"
            for class_id, class_name in class_names.items()
        ]
        # Mesmas tonalidades utilizadas nas nuvens de palavras
        colors = [
            CLASS_GRAY_COLORS[dataset][class_id]
            for class_id in class_names
        ]
        # Gráfico de barras
        bars = ax.bar(
            range(len(labels)),
            counts.values,
            color=colors,
            width=0.7,
            edgecolor="white"
        )
        # Quantidade e percentual acima das barras
        for bar, count, pct in zip(
            bars,
            counts.values,
            percentages.values
        ):
            ax.annotate(
                f"{count:,}\n({pct:.1f}%)",
                xy=(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height()
                ),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=9,
                color="#333333"
            )
        # Título de cada dataset
        ax.set_title(
            dataset.replace("_", " ").title(),
            fontsize=14,
            fontweight="bold",
            color="#333333"
        )
        ax.set_xlabel("Classes")
        ax.set_ylabel("Amostras")
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylim(
            0,
            max(counts.values) * 1.20
        )
        ax.grid(
            axis="y",
            alpha=0.2,
            color="#808080"
        )
        ax.set_axisbelow(True)
    # Título geral
    fig.suptitle(
        "Distribuição e balanceamento das classes",
        fontsize=17,
        fontweight="bold",
        color="#333333"
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    # Salvar figura
    filename = (
        OUTPUT_DIR /
        "distribuicao_classes_liar.jpg"
    )
    fig.savefig(
        filename,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )
    plt.show()
    plt.close(fig)
    print(f"Imagem salva: {filename}")

# Gerar figura
plot_class_distribution_all(df_liar)

In [ ]:
# ============================================================
# Pré-processamento para nuvens de palavras
# Etapas do pré-processamento: tratamento de valores nulos, conversão para minúsculas, remoção de URLs, números, pontuação,
#caracteres especiais, stopwords e palavras com menos de 3 caracteres, seguido de tokenização e reconstrução do texto.
# ============================================================

def clean_text(text):
    """Normaliza o texto para visualização das palavras."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    words = [
        word
        for word in text.split()
        if len(word) > 2 and word not in STOPWORDS
    ]
    return " ".join(words)
df_liar["clean_text"] = df_liar["statement"].map(clean_text)

## 7. NUVENS DE PALAVRAS POR CLASSE

In [ ]:
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# Função para gerar as nuvens de palavras

def plot_wordclouds(df, dataset, max_words=100):
    """
    Gera nuvens de palavras para cada classe do dataset.

    - Figura para as seis classes do LIAR.
    - Grade fixa de 2 linhas x 3 colunas.
    - Nuvens com dimensões padronizadas.
    - Um tom de cinza diferente para cada classe.
    - Todas as palavras da mesma classe possuem a mesma cor.
    - Exportação em JPG com 300 dpi.
    """

    class_names = CLASS_NAMES[dataset]

    # Configuração fixa para todos os datasets
    N_ROWS = 2
    N_COLS = 3
    FIG_SIZE = (15, 9)

    fig, axes = plt.subplots(
        N_ROWS,
        N_COLS,
        figsize=FIG_SIZE
    )

    axes = axes.flatten()

    for idx, (class_id, class_name) in enumerate(class_names.items()):

        ax = axes[idx]

        # Selecionar textos da classe
        texts = df.loc[
            df[dataset] == class_id,
            "clean_text"
        ]

        # Calcular frequência das palavras
        frequencies = Counter(
            word
            for text in texts
            for word in text.split()
        )

        # Obter o tom de cinza da classe
        gray_color = CLASS_GRAY_COLORS[dataset][class_id]

        if frequencies:

            cloud = WordCloud(
                width=700,
                height=500,
                background_color="white",
                max_words=max_words,
                random_state=SEED,
                collocations=False,

                # Todas as palavras desta classe
                # recebem o mesmo tom de cinza
                color_func=lambda *args, c=gray_color, **kwargs: c,

            ).generate_from_frequencies(frequencies)

            ax.imshow(
                cloud,
                interpolation="bilinear",
                aspect="equal"
            )

        else:
            ax.text(
                0.5, 0.5,
                "Sem palavras",
                ha="center",
                va="center"
            )

        # Título da classe
        ax.set_title(
            f"{class_id} — {class_name}\n"
            f"({len(texts):,} textos)",
            fontsize=12,
            fontweight="bold",
            color="#333333",
            pad=10
        )

        ax.axis("off")

    # Ocultar espaços não utilizados
    for idx in range(len(class_names), len(axes)):
        axes[idx].axis("off")

    # Título principal
    fig.suptitle(
        f"Word Clouds — {dataset}",
        fontsize=16,
        fontweight="bold",
        y=0.98
    )

    # Padronizar espaçamento
    fig.subplots_adjust(
        left=0.04,
        right=0.96,
        bottom=0.05,
        top=0.88,
        wspace=0.12,
        hspace=0.30
    )

    # Salvar as figuras
    OUTPUT_DIR = Path("figures")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    filename = OUTPUT_DIR / f"wordcloud_{dataset}.jpg"

    fig.savefig(
        filename,
        dpi=300,
        facecolor="white"
    )

    plt.show()
    plt.close(fig)

    print(f"Imagem salva: {filename}")

# Execução para os datasets
for dataset in ["liar"]:
    plot_wordclouds(df_liar, dataset)